# 07 — Corrective RAG (CRAG)

## Goal
Build a practical Corrective RAG pipeline over the CIS Microsoft Windows 11 Enterprise Benchmark. The key idea is to **evaluate retrieved evidence before generating an answer** and correct the retrieval when evidence looks weak.

This workshop version uses a transparent, lightweight relevance grader:
- Checks whether retrieved chunks contain query-related terms.
- Keeps relevant-looking chunks and reports the score.
- If evidence appears weak, runs a corrective search using a reformulated query.
- Generates an answer grounded in the resulting context, with source labels.

This is a teaching implementation, not a robust semantic grader. Keyword overlap is imperfect and should not be treated as proof of relevance. Benchmark text alone also does not prove a real Windows machine is compliant.

**Prerequisite:** Run `00b_ingestion_vector_store.ipynb` first.

## Architecture

```mermaid
flowchart TD
    A["User question"] --> B["Initial vector retrieval"]
    B --> C["Grade retrieved chunks"]
    C --> D{"Evidence relevant enough?"}
    D -->|"Yes"| E["Keep useful evidence"]
    D -->|"No"| F["Corrective query"]
    F --> G["Second retrieval"]
    G --> H["Merge and deduplicate"]
    E --> I["Build bounded context"]
    H --> I
    I --> J["LLM generates grounded answer"]
    J --> K["Answer + source references"]

    classDef input fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px;
    classDef retrieve fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:2px;
    classDef grade fill:#fef3c7,stroke:#d97706,color:#78350f,stroke-width:2px;
    classDef decision fill:#fce7f3,stroke:#db2777,color:#831843,stroke-width:2px;
    classDef answer fill:#ede9fe,stroke:#7c3aed,color:#3b0764,stroke-width:2px;
    class A,K input;
    class B,G,F retrieve;
    class C grade;
    class D decision;
    class E,H,I,J answer;
```

The Mermaid source is embedded in the notebook. Rendering depends on your notebook viewer.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import re
from collections import OrderedDict

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "vector_store").exists() and (PROJECT_ROOT.parent / "vector_store").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "chroma_cis_windows11"
COLLECTION_NAME = "cis_windows11_benchmark"

print("Project root:", PROJECT_ROOT)
print("Vector store exists:", VECTOR_STORE_DIR.exists())

## 2. Select a model provider

Select `ollama` or `groq`. For Groq, store the API key in `secrets/groq_api_key.txt` under the project root.

In [ ]:
PROVIDER = "ollama"  # "ollama" or "groq"
OLLAMA_MODEL = "gemma4:e2b"
GROQ_MODEL = "llama-3.3-70b-versatile"

if PROVIDER == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print(f"Using Ollama model: {OLLAMA_MODEL}")
elif PROVIDER == "groq":
    from langchain_groq import ChatGroq
    key_path = PROJECT_ROOT / "secrets" / "groq_api_key.txt"
    if not key_path.exists():
        raise FileNotFoundError(
            f"Groq key file not found: {key_path}. Create it and put only your API key in it."
        )
    groq_api_key = key_path.read_text(encoding="utf-8").strip()
    if not groq_api_key:
        raise ValueError(f"Groq key file is empty: {key_path}")
    llm = ChatGroq(model=GROQ_MODEL, api_key=groq_api_key, temperature=0)
    print(f"Using Groq model: {GROQ_MODEL}")
else:
    raise ValueError("PROVIDER must be 'ollama' or 'groq'")

## 3. Open the persisted Chroma store

Use the same embedding model as the ingestion notebook.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
if not VECTOR_STORE_DIR.exists():
    raise FileNotFoundError(
        f"Vector store not found at {VECTOR_STORE_DIR}. "
        "Run notebooks/00b_ingestion_vector_store.ipynb first."
    )

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
    embedding_function=embedding_model,
)
print("Connected to:", COLLECTION_NAME)
print("Stored record count:", vector_store._collection.count())

## 4. Define a simple relevance grader

The grader tokenizes the question and each chunk, removes common words, and measures overlap. The score is easy to inspect, but it misses synonyms and conceptual relevance. Treat it as a baseline to compare against an LLM-based grader later.

In [ ]:
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can", "does", "for",
    "from", "how", "in", "is", "it", "of", "on", "or", "the", "this", "to",
    "what", "when", "where", "which", "with", "would", "should", "explain",
    "recommend", "recommended", "benchmark"
}

def terms(text: str) -> set[str]:
    return {
        token for token in re.findall(r"[a-zA-Z0-9]+", text.lower())
        if len(token) > 2 and token not in STOP_WORDS
    }

def grade_document(question: str, document) -> dict:
    q_terms = terms(question)
    d_terms = terms(document.page_content)
    overlap = q_terms & d_terms
    score = len(overlap) / max(len(q_terms), 1)
    return {
        "score": score,
        "overlap_terms": sorted(overlap),
        "relevant": score >= 0.15,
    }

def grade_documents(question: str, documents: list) -> list[dict]:
    graded = []
    for doc in documents:
        grade = grade_document(question, doc)
        graded.append({"document": doc, **grade})
    return graded

## 5. Initial retrieval and corrective search

The pipeline retrieves an initial set, grades it, and keeps chunks that pass the threshold. If too few chunks pass, it tries one corrective query that preserves the user's intent while adding benchmark/security context.

In [ ]:
def doc_identity(doc):
    meta = doc.metadata or {}
    return (
        str(meta.get("source", "unknown")),
        meta.get("page"),
        meta.get("start_index"),
        doc.page_content[:180],
    )

def deduplicate_documents(documents):
    unique = OrderedDict()
    for doc in documents:
        unique.setdefault(doc_identity(doc), doc)
    return list(unique.values())

def corrective_query(question: str) -> str:
    return (
        "CIS Microsoft Windows 11 Enterprise security benchmark recommendation "
        "configuration requirement and assessment for: " + question
    )

def corrective_retrieve(question: str, k: int = 6, min_relevant: int = 2) -> dict:
    initial_docs = vector_store.similarity_search(question, k=k)
    initial_grades = grade_documents(question, initial_docs)
    kept = [item["document"] for item in initial_grades if item["relevant"]]
    correction_used = False
    correction_docs = []
    correction_grades = []

    if len(kept) < min_relevant:
        correction_used = True
        query2 = corrective_query(question)
        correction_docs = vector_store.similarity_search(query2, k=k)
        correction_grades = grade_documents(question, correction_docs)
        kept.extend(item["document"] for item in correction_grades if item["relevant"])

    final_docs = deduplicate_documents(kept)
    return {
        "initial_documents": initial_docs,
        "initial_grades": initial_grades,
        "correction_used": correction_used,
        "corrective_query": corrective_query(question) if correction_used else None,
        "correction_documents": correction_docs,
        "correction_grades": correction_grades,
        "documents": final_docs[:k],
        "final_grades": grade_documents(question, final_docs[:k]),
    }

## 6. Build context and generate a grounded answer

Source labels let you inspect which chunks support the answer. The model is explicitly instructed not to infer actual machine compliance from benchmark recommendations.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def source_label(doc, index: int) -> str:
    meta = doc.metadata or {}
    source = Path(str(meta.get("source", "unknown source"))).name
    page = meta.get("page", "unknown")
    start = meta.get("start_index")
    label = f"[S{index}] {source}, page {page}"
    if start is not None:
        label += f", chunk start {start}"
    return label

def build_context(documents, max_chars_per_doc: int = 1800) -> str:
    return "\n\n---\n\n".join(
        f"{source_label(doc, i)}\n{doc.page_content.strip()[:max_chars_per_doc]}"
        for i, doc in enumerate(documents, start=1)
    )

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You explain a Windows 11 security benchmark. Answer using only the supplied context. "
     "Cite claims using source labels such as [S1]. If context is insufficient, say so. "
     "Do not claim that a real machine is compliant based only on benchmark text. "
     "Distinguish benchmark recommendations from evidence of actual machine settings."),
    ("human", "Question: {question}\n\nRetrieved context:\n{context}")
])

def corrective_rag(question: str, k: int = 6) -> dict:
    retrieval = corrective_retrieve(question, k=k)
    context = build_context(retrieval["documents"])
    response = (answer_prompt | llm).invoke({
        "question": question,
        "context": context or "[No relevant chunks passed the baseline grader.]",
    })
    answer = response.content if hasattr(response, "content") else str(response)
    return {**retrieval, "context": context, "answer": answer}

## 7. Run a question

In [ ]:
question = "What does the benchmark recommend about screen locking?"
result = corrective_rag(question, k=6)

print("Correction used:", result["correction_used"])
print("Corrective query:", result["corrective_query"])
print("Initial chunks:", len(result["initial_documents"]))
print("Chunks after grading:", len(result["documents"]))
print("\nANSWER\n")
print(result["answer"])

## 8. Inspect the relevance grades

Review overlap terms and scores. A chunk may be relevant even with low lexical overlap, or irrelevant despite sharing a few terms.

In [ ]:
print("INITIAL RETRIEVAL GRADES")
for i, item in enumerate(result["initial_grades"], start=1):
    doc = item["document"]
    print("-" * 88)
    print(f"{source_label(doc, i)} | score={item['score']:.3f} | relevant={item['relevant']}")
    print("Overlap:", item["overlap_terms"])
    print(doc.page_content[:500])

if result["correction_used"]:
    print("\nCORRECTIVE RETRIEVAL GRADES")
    for i, item in enumerate(result["correction_grades"], start=1):
        print("-" * 88)
        print(f"{source_label(item['document'], i)} | score={item['score']:.3f} | relevant={item['relevant']}")
        print("Overlap:", item["overlap_terms"])
        print(item["document"].page_content[:500])

## 9. Try several questions

Observe whether the baseline grader triggers corrective retrieval. Adjust `k`, `min_relevant`, or the relevance threshold in `grade_document()` and compare the behavior.

In [ ]:
test_questions = [
    "What is the recommended screen lock setting?",
    "How should account lockout be configured?",
    "Which recommendations concern password policy and auditing?",
]
for q in test_questions:
    r = corrective_retrieve(q, k=6)
    print("\nQUESTION:", q)
    print("Correction used:", r["correction_used"])
    print("Initial relevant:", sum(x["relevant"] for x in r["initial_grades"]))
    print("Final chunks:", len(r["documents"]))
    print("Corrective query:", r["corrective_query"])

## 10. Lightweight checks

In [ ]:
# Check grader mechanics with a tiny mock document-like object.
class MockDocument:
    def __init__(self, page_content):
        self.page_content = page_content
        self.metadata = {}

mock = MockDocument("Windows password policy requires a suitable password length.")
grade = grade_document("Windows password policy", mock)
assert grade["score"] > 0
assert grade["overlap_terms"]
assert deduplicate_documents([]) == []
print("Basic grader/helper checks passed.")

## Summary and limitations

This notebook demonstrates:
- Retrieval grading before answer generation.
- Corrective retrieval when too few chunks pass the baseline grader.
- Source inspection and grounded response generation.

Important limitations:
1. Keyword overlap is not semantic relevance. Synonyms, abbreviations, and paraphrases may be missed.
2. The threshold is arbitrary and should be tuned against a labelled evaluation set.
3. The corrective query may not improve retrieval; compare initial and corrective chunks.
4. A chunk passing the grader does not guarantee it supports every claim in the final answer.
5. For a stronger version, replace the keyword grader with a structured LLM relevance grader, validate its output, and evaluate retrieval recall and answer faithfulness.
6. Benchmark recommendations alone do not demonstrate a real endpoint's current configuration or compliance.